<a href="https://colab.research.google.com/github/mikilezen/disease-classification/blob/main/model%20training%20code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
import pandas as pd
new = []
df = pd.read_csv('/content/symptom_specialty_90k_v2.csv')
for col in range(0, len(df.specialties)):
  newcol = df.specialties[col].split('|')[1-1]
  new.append(newcol)
new = pd.Series(new)
df['specialties'] = new
print(df.head())
df['combined_text'] = df['symptom'] + ' ' + df['description']
print(df.head())

   id          symptom    specialties  \
0   1   hallucinations     Psychiatry   
1   2   blurred vision  Ophthalmology   
2   3         red eyes  Ophthalmology   
3   4   hallucinations     Psychiatry   
4   5  chest tightness    Pulmonology   

                                         description  
0  I have been suffering from hallucinations for ...  
1  I suddenly started having blurred vision and I...  
2  Every day, I deal with red eyes, and it makes ...  
3  I suddenly started having hallucinations and I...  
4  For some time now, I feel chest tightness, esp...  
   id          symptom    specialties  \
0   1   hallucinations     Psychiatry   
1   2   blurred vision  Ophthalmology   
2   3         red eyes  Ophthalmology   
3   4   hallucinations     Psychiatry   
4   5  chest tightness    Pulmonology   

                                         description  \
0  I have been suffering from hallucinations for ...   
1  I suddenly started having blurred vision and I...   
2  Every

In [36]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import string

# Download necessary NLTK data (if not already downloaded)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab') # Added to resolve LookupError

# Initialize lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Ensure text is a string to prevent AttributeError for floats (NaNs)
    text = str(text)
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords and lemmatize
    processed_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(processed_tokens)

# Apply preprocessing to the combined_text column
# Ensure the column is explicitly converted to string type to handle potential NaN values
df['processed_text'] = df['combined_text'].astype(str).apply(preprocess_text)

print(df.head())

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


   id          symptom    specialties  \
0   1   hallucinations     Psychiatry   
1   2   blurred vision  Ophthalmology   
2   3         red eyes  Ophthalmology   
3   4   hallucinations     Psychiatry   
4   5  chest tightness    Pulmonology   

                                         description  \
0  I have been suffering from hallucinations for ...   
1  I suddenly started having blurred vision and I...   
2  Every day, I deal with red eyes, and it makes ...   
3  I suddenly started having hallucinations and I...   
4  For some time now, I feel chest tightness, esp...   

                                       combined_text  \
0  hallucinations I have been suffering from hall...   
1  blurred vision I suddenly started having blurr...   
2  red eyes Every day, I deal with red eyes, and ...   
3  hallucinations I suddenly started having hallu...   
4  chest tightness For some time now, I feel ches...   

                                      processed_text  
0  hallucination sufferi

**Reasoning**:
With the text preprocessing complete, the next step in the data preparation is to encode the categorical 'specialties' column into numerical labels, which is a necessary step before training any machine learning model. This will be done using `LabelEncoder` from `sklearn.preprocessing` as specified in the instructions.



In [37]:
from sklearn.preprocessing import LabelEncoder

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Encode the 'specialties' column
df['specialties_encoded'] = label_encoder.fit_transform(df['specialties'])

print(df[['specialties', 'specialties_encoded']].head())
print(f"Number of unique specialties: {len(label_encoder.classes_)}")

     specialties  specialties_encoded
0     Psychiatry                    8
1  Ophthalmology                    6
2  Ophthalmology                    6
3     Psychiatry                    8
4    Pulmonology                    9
Number of unique specialties: 10


**Reasoning**:
Now that the text has been preprocessed and the target labels encoded, the next step is to vectorize the processed text using `TfidfVectorizer`. This will convert the text data into a numerical format suitable for machine learning models, as specified in the instructions.



In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limiting features for manageability

# Fit and transform the processed text
X = tfidf_vectorizer.fit_transform(df['processed_text'])

print(f"Shape of TF-IDF matrix: {X.shape}")

Shape of TF-IDF matrix: (90000, 93)


**Reasoning**:
With the text vectorized and target labels encoded, the final step in data preprocessing is to split the data into training and testing sets. This will be done using `train_test_split` to prepare the data for model development and evaluation, as specified in the instructions.



In [39]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
y = df['specialties_encoded']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (72000, 93)
Shape of X_test: (18000, 93)
Shape of y_train: (72000,)
Shape of y_test: (18000,)


**Reasoning**:
To begin training the classification model as per the instructions, I need to import the `LogisticRegression` classifier from `sklearn.linear_model`, instantiate it with specified parameters for better convergence and handling multi-class data, and then train the model using the preprocessed training data (`X_train` and `y_train`).



In [40]:
from sklearn.linear_model import LogisticRegression

# Initialize Logistic Regression model
# Increased max_iter for convergence on larger datasets
# saga solver is good for multiclass and can handle large datasets (sparse data with L1/L2 regularization)
model = LogisticRegression(max_iter=1000, solver='saga', multi_class='multinomial', n_jobs=-1)

# Train the model
model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Logistic Regression model trained successfully.


**Reasoning**:
The previous code produced a `FutureWarning` regarding the `multi_class` parameter in `LogisticRegression`, stating that it's deprecated and will default to 'multinomial' in future versions. Since 'multinomial' is the default and only supported option for the 'saga' solver in multi-class classification, explicitly setting `multi_class='multinomial'` is unnecessary. Removing this parameter will resolve the warning without changing the intended behavior.



In [41]:
from sklearn.linear_model import LogisticRegression

# Initialize Logistic Regression model
# Increased max_iter for convergence on larger datasets
# saga solver is good for multiclass and can handle large datasets (sparse data with L1/L2 regularization)
model = LogisticRegression(max_iter=1000, solver='saga', n_jobs=-1)

# Train the model
model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


## Evaluate Model Performance

### Subtask:
Evaluate the trained model's performance on the test set. This will involve calculating key metrics such as accuracy, precision, recall, and F1-score. A classification report will be generated to provide a detailed breakdown of the model's performance for each specialty.


**Reasoning**:
To evaluate the model's performance, I need to import the `classification_report` and `accuracy_score` functions from `sklearn.metrics`, use the trained model to predict on the test set, and then print both the classification report and the overall accuracy.



In [42]:
from sklearn.metrics import classification_report, accuracy_score

# Make predictions on the test set
y_pred = model.predict(X_test)

# Print the classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Print the overall accuracy score
accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {accuracy:.4f}")

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1812
           1       1.00      1.00      1.00      1728
           2       1.00      1.00      1.00      1797
           3       1.00      1.00      1.00      1866
           4       0.93      0.88      0.90      1780
           5       1.00      1.00      1.00      1768
           6       1.00      1.00      1.00      1778
           7       0.89      0.93      0.91      1851
           8       1.00      1.00      1.00      1756
           9       1.00      1.00      1.00      1864

    accuracy                           0.98     18000
   macro avg       0.98      0.98      0.98     18000
weighted avg       0.98      0.98      0.98     18000

Overall Accuracy: 0.9810


In [34]:
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import string

# Assuming NLTK data is already downloaded in the environment from previous steps.
# If not, uncomment these lines:
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')

# Initialize lemmatizer and stopwords (must be done each time if kernel resets)
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords and lemmatize
    processed_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(processed_tokens)


# Load the saved model (using the Logistic Regression model for this example,
# but you could load xgb_model as well if saved)
loaded_model = joblib.load('logistic_regression_model.joblib')
print("Logistic Regression model loaded successfully.")

# Load the saved TF-IDF vectorizer
loaded_tfidf_vectorizer = joblib.load('tfidf_vectorizer.joblib')
print("TF-IDF vectorizer loaded successfully.")

# Load the saved LabelEncoder
loaded_label_encoder = joblib.load('label_encoder.joblib')
print("Label encoder loaded successfully.")

# Define a new sample input
new_sample_input = "I have a severe cough and difficulty breathing."
print(f"\nNew sample input: '{new_sample_input}'")

# Preprocess the new sample input
processed_new_sample_input = preprocess_text(new_sample_input)
print(f"Processed new sample input: '{processed_new_sample_input}'")

# Transform the processed sample input using the loaded TF-IDF vectorizer
new_sample_input_vectorized = loaded_tfidf_vectorizer.transform([processed_new_sample_input])
print(f"Shape of vectorized new sample input: {new_sample_input_vectorized.shape}")

# Predict the encoded specialty using the loaded model
predicted_encoded_specialty = loaded_model.predict(new_sample_input_vectorized)

# Inverse transform the predicted encoded specialty to get the human-readable specialty
predicted_specialty = loaded_label_encoder.inverse_transform(predicted_encoded_specialty)

print(f"Predicted medical specialty: {predicted_specialty[0]}")

Logistic Regression model loaded successfully.
TF-IDF vectorizer loaded successfully.
Label encoder loaded successfully.

New sample input: 'I have a severe cough and difficulty breathing.'
Processed new sample input: 'severe cough difficulty breathing'
Shape of vectorized new sample input: (1, 94)
Predicted medical specialty: Pulmonology


**Reasoning**:
The previous output showed `UndefinedMetricWarning` due to zero division in precision calculation for some classes in the classification report. To address this, I will set the `zero_division` parameter to 0 in `classification_report()` so that precision, recall, and f1-score for classes with no predicted samples are set to 0.0.



In [43]:
from sklearn.metrics import classification_report, accuracy_score

# Make predictions on the test set
y_pred = model.predict(X_test)

# Print the classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# Print the overall accuracy score
accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {accuracy:.4f}")

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1812
           1       1.00      1.00      1.00      1728
           2       1.00      1.00      1.00      1797
           3       1.00      1.00      1.00      1866
           4       0.93      0.88      0.90      1780
           5       1.00      1.00      1.00      1768
           6       1.00      1.00      1.00      1778
           7       0.89      0.93      0.91      1851
           8       1.00      1.00      1.00      1756
           9       1.00      1.00      1.00      1864

    accuracy                           0.98     18000
   macro avg       0.98      0.98      0.98     18000
weighted avg       0.98      0.98      0.98     18000

Overall Accuracy: 0.9810


## Prepare Model for Production

### Subtask:
Save the trained machine learning model to disk using a standard serialization method (like `pickle` or `joblib`). This allows the model to be easily loaded and used for making predictions on new, unseen data in a production environment. An example of loading the model and making a prediction will be provided.


**Reasoning**:
To prepare the model for production, I will first import the `joblib` library. Then, I will save the trained Logistic Regression model, the `TfidfVectorizer`, and the `LabelEncoder` to disk using `joblib.dump()`. This ensures that all necessary components for making predictions are preserved.



In [44]:
import joblib

# Save the trained model with .joblib extension
joblib.dump(model, 'logistic_regression_model.joblib')
print("Logistic Regression model saved successfully as .joblib")

# Save the TF-IDF vectorizer with .joblib extension
joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.joblib')
print("TF-IDF vectorizer saved successfully as .joblib")

# Save the LabelEncoder with .joblib extension
joblib.dump(label_encoder, 'label_encoder.joblib')
print("Label encoder saved successfully as .joblib")

Logistic Regression model saved successfully as .joblib
TF-IDF vectorizer saved successfully as .joblib
Label encoder saved successfully as .joblib


**Reasoning**:
Now that the model components are saved, I will load them back to demonstrate how they can be used for making predictions on new data, as specified in the instructions. This involves loading the model, TF-IDF vectorizer, and label encoder, then using them to preprocess a sample input and predict its specialty.



In [45]:
import joblib

# Load the saved model
loaded_model = joblib.load('logistic_regression_model.joblib')
print("Logistic Regression model loaded successfully.")

# Load the saved TF-IDF vectorizer
loaded_tfidf_vectorizer = joblib.load('tfidf_vectorizer.joblib')
print("TF-IDF vectorizer loaded successfully.")

# Load the saved LabelEncoder
loaded_label_encoder = joblib.load('label_encoder.joblib')
print("Label encoder loaded successfully.")

# Define a sample input
sample_input = "I have a headache and feel dizzy."
print(f"\nSample input: '{sample_input}'")

# Preprocess the sample input using the same preprocess_text function
# (Assuming preprocess_text is available in the current kernel state)
processed_sample_input = preprocess_text(sample_input)
print(f"Processed sample input: '{processed_sample_input}'")

# Transform the processed sample input using the loaded TF-IDF vectorizer
sample_input_vectorized = loaded_tfidf_vectorizer.transform([processed_sample_input])
print(f"Shape of vectorized sample input: {sample_input_vectorized.shape}")

# Predict the encoded specialty using the loaded model
predicted_encoded_specialty = loaded_model.predict(sample_input_vectorized)

# Inverse transform the predicted encoded specialty to get the human-readable specialty
predicted_specialty = loaded_label_encoder.inverse_transform(predicted_encoded_specialty)

print(f"Predicted medical specialty: {predicted_specialty[0]}")

Logistic Regression model loaded successfully.
TF-IDF vectorizer loaded successfully.
Label encoder loaded successfully.

Sample input: 'I have a headache and feel dizzy.'
Processed sample input: 'headache feel dizzy'
Shape of vectorized sample input: (1, 93)
Predicted medical specialty: Neurology


## Final Task

### Subtask:
Summarize the model's performance, discuss its readiness for production, and highlight any significant findings or next steps from this classification task.


In [46]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TfidfVectorizer with ngram_range=(1, 2) to capture unigrams and bigrams
# Removed max_features to allow for a richer feature set initially
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2))

# Fit and transform the processed text
X = tfidf_vectorizer.fit_transform(df['processed_text'])

print(f"Shape of TF-IDF matrix with n-gram range (1,2): {X.shape}")

Shape of TF-IDF matrix with n-gram range (1,2): (90000, 868)


In [47]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
# X is already updated from the previous step
y = df['specialties_encoded']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (72000, 868)
Shape of X_test: (18000, 868)
Shape of y_train: (72000,)
Shape of y_test: (18000,)


In [ ]:
xgb_model.fit(X_train, y_train)
print("XGBoost model trained successfully on the updated data.")

In [ ]:
from xgboost import XGBClassifier

# Initialize XGBClassifier model
# num_class is set to the number of unique specialties
# objective='multi:softmax' for multi-class classification outputs the predicted class directly
# use_label_encoder=False to suppress a deprecation warning
# eval_metric='mlogloss' is a common evaluation metric for multi-class classification
xgb_model = XGBClassifier(
    objective='multi:softmax',
    num_class=len(label_encoder.classes_),
    eval_metric='mlogloss',
    use_label_encoder=False, # Suppress warning
    n_jobs=-1, # Use all available CPU cores
    random_state=42 # For reproducibility
)

# Train the model
xgb_model.fit(X_train, y_train)

print("XGBoost model trained successfully.")

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
# Initialize XGBClassifier model
# num_class is set to the number of unique specialties
# objective='multi:softmax' for multi-class classification outputs the predicted class directly
# eval_metric='mlogloss' is a common evaluation metric for multi-class classification
xgb_model = XGBClassifier(
    objective='multi:softmax',
    num_class=len(label_encoder.classes_),
    eval_metric='mlogloss',
    n_jobs=-1, # Use all available CPU cores
    random_state=42 # For reproducibility
)

# Train the model
xgb_model.fit(X_train, y_train)

print("XGBoost model trained successfully.")
prd = xgb_model.predict(X_test)
print(f"XGBoost Accuracy: {accuracy_score(y_test, prd)*100:.2f}%")


In [ ]:
requirements = """
fastapi
uvicorn
joblib
nltk
scikit-learn
pydantic
python-multipart
"""
with open('requirements.txt', 'w') as f:
    f.write(requirements.strip())
print('requirements.txt created successfully.')

In [ ]:
app_code = """
import joblib
from fastapi import FastAPI
from pydantic import BaseModel
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import string
from contextlib import asynccontextmanager

# Helper function for text preprocessing
def preprocess_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))
    processed_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(processed_tokens)

# Global dictionary to store model artifacts
artifacts = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Load the model, TF-IDF vectorizer, and label encoder on startup with .joblib extension
    print("Loading model artifacts...")
    artifacts['model'] = joblib.load('logistic_regression_model.joblib')
    artifacts['tfidf_vectorizer'] = joblib.load('tfidf_vectorizer.joblib')
    artifacts['label_encoder'] = joblib.load('label_encoder.joblib')

    # Pre-download NLTK data
    nltk.download('punkt')
    nltk.download('stopwords')
    nltk.download('wordnet')
    nltk.download('punkt_tab')

    print("Model artifacts loaded successfully.")
    yield
    # Clean up artifacts on shutdown
    artifacts.clear()

app = FastAPI(lifespan=lifespan)

class PredictionRequest(BaseModel):
    text: str

@app.get("/health")
async def health_check():
    return {"status": "ok", "model_loaded": 'model' in artifacts}

@app.post("/predict")
async def predict_specialty(request: PredictionRequest):
    processed_text = preprocess_text(request.text)
    vectorized_text = artifacts['tfidf_vectorizer'].transform([processed_text])
    predicted_encoded = artifacts['model'].predict(vectorized_text)
    predicted_specialty = artifacts['label_encoder'].inverse_transform(predicted_encoded)
    return {"predicted_specialty": predicted_specialty[0]}
"""

with open('app.py', 'w') as f:
    f.write(app_code.strip())
print('app.py updated with .joblib extensions successfully.')

In [ ]:
import joblib
from fastapi import FastAPI
from pydantic import BaseModel
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import string

# Initialize FastAPI app
app = FastAPI()

# Global variables for model, vectorizer, and encoder
model = None
tfidf_vectorizer = None
label_encoder = None
lemmatizer = None
stop_words = None

# Helper function for text preprocessing (same as used during training)
def preprocess_text(text):
    global lemmatizer, stop_words
    if lemmatizer is None:
        # Download necessary NLTK data (if not already downloaded)
        try:
            nltk.data.find('tokenizers/punkt')
        except nltk.downloader.DownloadError:
            nltk.download('punkt')
        try:
            nltk.data.find('corpora/stopwords')
        except nltk.downloader.DownloadError:
            nltk.download('stopwords')
        try:
            nltk.data.find('corpora/wordnet')
        except nltk.downloader.DownloadError:
            nltk.download('wordnet')
        try:
            nltk.data.find('tokenizers/punkt_tab')
        except nltk.downloader.DownloadError:
            nltk.download('punkt_tab')

        lemmatizer = WordNetLemmatizer()
        stop_words = set(stopwords.words('english'))

    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    processed_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(processed_tokens)

# Pydantic model for request body
class PredictionRequest(BaseModel):
    text: str

@app.on_event("startup")
async def load_artifacts():
    """Load the model, TF-IDF vectorizer, and label encoder on startup."""
    global model, tfidf_vectorizer, label_encoder
    print("Loading model artifacts...")
    model = joblib.load('logistic_regression_model.joblib')
    tfidf_vectorizer = joblib.load('tfidf_vectorizer.joblib')
    label_encoder = joblib.load('label_encoder.joblib')
    print("Model artifacts loaded successfully.")

@app.get("/health")
async def health_check():
    """Health check endpoint."""
    return {"status": "ok", "model_loaded": model is not None}

@app.post("/predict")
async def predict_specialty(request: PredictionRequest):
    """Predicts the medical specialty based on input text."""
    if model is None or tfidf_vectorizer is None or label_encoder is None:
        return {"error": "Model artifacts not loaded. Please try again or check logs."}, 500

    # Preprocess the input text
    processed_text = preprocess_text(request.text)

    # Transform the processed text using the loaded TF-IDF vectorizer
    vectorized_text = tfidf_vectorizer.transform([processed_text])

    # Predict the encoded specialty
    predicted_encoded_specialty = model.predict(vectorized_text)

    # Inverse transform to get the human-readable specialty
    predicted_specialty = label_encoder.inverse_transform(predicted_encoded_specialty)

    return {"predicted_specialty": predicted_specialty[0]}

# To run this API locally:
# 1. Save the code above as `app.py`.
# 2. Make sure `logistic_regression_model.joblib`, `tfidf_vectorizer.joblib`, and `label_encoder.joblib` are in the same directory.
# 3. Install uvicorn and fastapi: `pip install uvicorn fastapi python-multipart joblib nltk scikit-learn`
# 4. Run from your terminal: `uvicorn app:app --host 0.0.0.0 --port 8000 --reload`
# 5. Open your browser to `http://0.0.0.0:8000/docs` to see the OpenAPI (Swagger) UI.

This code sets up a basic FastAPI application with:

*   An `@app.on_event("startup")` function to load the trained model, TF-IDF vectorizer, and label encoder when the application starts.
*   A `/health` endpoint to check if the API is running and if the model is loaded.
*   A `/predict` endpoint that accepts a JSON body with a `text` field, preprocesses it, vectorizes it, and returns the predicted medical specialty.

To run this locally:

1.  Save the code above as `app.py` in the same directory where your `.joblib` files (`logistic_regression_model.joblib`, `tfidf_vectorizer.joblib`, and `label_encoder.joblib`) are saved.
2.  Install the necessary Python packages. I'll provide the `requirements.txt` next.
3.  Open your terminal in that directory and run: `uvicorn app:app --host 0.0.0.0 --port 8000 --reload`.
4.  You can then access the interactive API documentation (Swagger UI) at `http://0.0.0.0:8000/docs` in your web browser to test the `/predict` endpoint.